Case Study: Patient readmission prediction

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline



In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve
)


In [10]:
data = {
    "age": [65, 45, 72, 34, 58, 80, 50, 39, 67, 55,
            70, 42, 61, 76, 48, 63, 52, 81, 37, 69],

    "heart_rate": [95, 82, 105, 75, 90, 110, 88, 78, 98, 92,
                   108, 80, 96, 112, 85, 100, 89, 115, 76, 102],

    "blood_pressure": [145, 120, 160, 115, 135, 170, 128, 118, 150, 140,
                       165, 122, 142, 175, 125, 155, 130, 180, 110, 148],

    "prior_visits": [5, 1, 6, 0, 3, 8, 2, 1, 5, 4,
                     7, 1, 4, 9, 2, 6, 3, 10, 0, 5],

    "diagnosis": [
        "Diabetes", "Infection", "HeartDisease", "Healthy",
        "Diabetes", "HeartDisease", "Infection", "Healthy",
        "Diabetes", "HeartDisease",
        "HeartDisease", "Infection", "Diabetes", "HeartDisease",
        "Healthy", "Diabetes", "Infection", "HeartDisease",
        "Healthy", "Diabetes"
    ],

    # 1 = readmitted within 30 days
    # 0 = not readmitted
    "readmitted": [
        1, 0, 1, 0, 1, 1, 0, 0, 1, 1,
        1, 0, 1, 1, 0, 1, 0, 1, 0, 1
    ]
}

In [ ]:
df=pd.DataFrame(data)
print("Patient Dataset:")
print(df)

In [12]:
X=df.drop("readmitted", axis=1)
y=df["readmitted"]

In [13]:
numerical_features = [
    "age",
    "heart_rate",
    "blood_pressure",
    "prior_visits"
]

categorical_features = [
    "diagnosis"
]

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

In [15]:
model = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000
)

pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", model)
    ]
)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [ ]:
pipeline.fit(X_train, y_train)

print("\nModel training completed.")

In [ ]:
y_probability = pipeline.predict_proba(X_test)[:, 1]

print("\nPredicted Readmission Probabilities:")
print(y_probability)

y_pred = pipeline.predict(X_test)

print("\nPredicted Classes:")
print(y_pred)

In [ ]:
roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print("\nROC-AUC Score:", round(roc_auc, 4))

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


In [ ]:
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)



In [ ]:
fpr, tpr, thresholds = roc_curve(
    y_test,
    y_probability
)

plt.figure(figsize=(7, 5))

plt.plot(
    fpr,
    tpr,
    label=f"Logistic Regression (AUC = {roc_auc:.2f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Hospital Readmission Prediction")

plt.legend()
plt.grid()

plt.show()

In [ ]:
results = X_test.copy()

results["Actual"] = y_test.values
results["Predicted"] = y_pred
results["Readmission_Probability"] = y_probability

print("\nPrediction Results:")
print(results)